# Step 1. Create list of words

**From Oxford 3000 and 5000**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

dataset_s = "oxford3000-5000.csv"
dataset_url_s = "https://www.oxfordlearnersdictionaries.com/wordlists/oxford3000-5000"
home_url_s = "https://www.oxfordlearnersdictionaries.com"

##### CRAWLING DATA #####

page = requests.get(dataset_url_s, headers={"User-agent": "Mozilla/5.0"})
soup = BeautifulSoup(page.text, "html.parser")
rows_list = []
id_num = 0
for entry in soup.select("#wordlistsContentPanel li"):
    dic = {}
    dic["id"] = id_num
    id_num = id_num + 1
    dic["word"] = entry["data-hw"]
    try:
        dic["url"] = home_url_s + entry.a["href"]
    except:
        dic["url"] = None
    rows_list.append(dic)
df = pd.DataFrame(rows_list)

##### SAVING DATA #####

df.to_csv(dataset_s, index=False)

**From Oxford Phrasal Academic Lexicon (OPAL)**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

dataset_s = "opal.csv"
dataset_url_s = "https://www.oxfordlearnersdictionaries.com/wordlists/opal"
home_url_s = "https://www.oxfordlearnersdictionaries.com"

##### CRAWLING DATA #####

page = requests.get(dataset_url_s, headers={"User-agent": "Mozilla/5.0"})
soup = BeautifulSoup(page.text, "html.parser")
rows_list = []
id_num = 0
for entry in soup.select("#wordlistsContentPanel li"):
    dic = {}
    dic["id"] = id_num
    id_num = id_num + 1
    dic["word"] = entry["data-hw"]
    try:
        dic["url"] = home_url_s + entry.a["href"]
    except:
        dic["url"] = None
    rows_list.append(dic)
df = pd.DataFrame(rows_list)

##### SAVING DATA #####

df.to_csv(dataset_s, index=False)

**From Oxford Phrase List**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

dataset_s = "oxford-phrase-list.csv"
dataset_url_s = "https://www.oxfordlearnersdictionaries.com/wordlists/oxford-phrase-list"
home_url_s = "https://www.oxfordlearnersdictionaries.com"

##### CRAWLING DATA #####

page = requests.get(dataset_url_s, headers={"User-agent": "Mozilla/5.0"})
soup = BeautifulSoup(page.text, "html.parser")
rows_list = []
id_num = 0
for entry in soup.select("#wordlistsContentPanel li"):
    dic = {}
    dic["id"] = id_num
    id_num = id_num + 1
    dic["word"] = entry["data-hw"]
    try:
        dic["url"] = home_url_s + entry.a["href"]
    except:
        dic["url"] = None
    rows_list.append(dic)
df = pd.DataFrame(rows_list)

##### SAVING DATA #####

df.to_csv(dataset_s, index=False)

# Step 2. Word of the day

**Word of the day - from Oxford**

In [ ]:
import urllib.request
import xmltodict
import pandas as pd

url = "https://www.oxfordlearnersdictionaries.com/wotd/wotdrss.xml"

##### FETCHING LIST OF WORDS #####

word_list = []

req = urllib.request.Request(url)
req.add_header("User-agent", "Mozilla/5.0")

with urllib.request.urlopen(req) as f:
    data = xmltodict.parse(f.read().decode("utf-8"))
    for word in data["feed"]["entry"]:
        dic = {}
        dic["title"] = word["title"]
        dic["url"] = word["link"]["@href"]
        dic["summary"] = word["summary"]
        word_list.append(dic)
        
df = pd.DataFrame(word_list)

##### SAVING LIST OF WORDS #####

df.to_csv("wotd.csv", index=False)

In [ ]:
import asyncio
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

##### LOADING LIST OF WORDS #####

df = pd.read_csv("wotd.csv")

##### DISPLAYING WORDS #####

def display_word(title, url, summary):
    future = asyncio.Future()
    
    clear_output()
    
    title_lable = widgets.Label(value = r"\(\Large \bf {" + title + "}\)")
    summary_lable = widgets.Label(value = summary)
    
    url_button = widgets.Button(description = "Show URL")
    url_button.on_click(lambda b: print(url))
    
    next_button = widgets.Button(description = "Next")
    def wait_on_click(change):
        future.set_result(change.description)
        next_button.on_click(wait_on_click, remove=True)
    next_button.on_click(wait_on_click)
    
    display_box =  widgets.VBox([title_lable, summary_lable, widgets.HBox([url_button, next_button])])
    display_box.layout.align_items = "center"
    
    display(display_box)
    return future

async def display_wotd():
    for index, row in df.iterrows():
        await display_word(row["title"], row["url"], row["summary"])
    print("Finish!")
        
asyncio.create_task(display_wotd())

# Step 3. Learn words from word lists

* Short term memory:
     * Capacity: 5-9 items
     * Duration: 18-30 seconds

* To determine how much the retrievability and stability of an item in long term memory, we must eliminate the effect of short term memory.

* Therefore, in one learning time, we must learn at least 10 items at the same time. Because the short term memory does not last in a long time, we can ignore the duration.

* Long term memory: The relationship between the retrievability $R$ and stability $S$ of memory is expressed by the following equation.
$$R = e^{-\frac{t}{S}}$$

* Idea: The current learning result is used to evaluate $R_c$ based on how difficult an item is learned. Then, we use it to evaluate the previous stability $S_p$ and estimate the current stability $S_c$ by the following equation.
$$S_c = S_p * \theta$$
with $\theta$ is an unknown constant.

In [ ]:
import datetime, random
import pandas as pd

##### PARAMETERS #####

retention = 0.9
initial_stability = 1
min_learning_items = 10
min_consecutive_success = 2


##### LOAD WORD LIST #####

wl = {}
wl_names = ["oxford3000-5000", "opal", "oxford-phrase-list"]

for name in wl_names:
    wl[name] = pd.read_csv(name + ".csv")
    
##### LEARNING LOG #####

learning_log = None
try:
    learning_log = pd.read_csv("learning_log.csv")
except:
    learning_log = pd.DataFrame(columns = ["wl_name", "wl_id",
                                           "R", "S", "date",
                                           "prev_R", "prev_S", "prev_date",
                                           "next_date"])
    learning_log.to_csv("learning_log.csv", index=False)

##### GET ALL ITEMS NEED TO REVIEW #####

now_date = datetime.datetime.now().date()
review_list = []
for index, item in learning_log.iterrows():
    if now_date <= datetime.date.fromisoformat(item["next_date"]):
        review_list.append(item.to_dict())
        
##### ADD NEW ITEMS TO LEARN #####

while len(review_list) < min_learning_items:
    random_wl_name = random.choice(wl_names)
    size_wl = len(wl[random_wl_name])
    random_wl_id = random.randrange(size_wl)
    
    if not ((learning_log["wl_name"] == random_wl_name) & (learning_log["wl_id"] == random_wl_id)).any():
        item = {}
        item["wl_name"] = random_wl_name
        item["wl_id"] = random_wl_id
        item["R"] = None
        item["S"] = None
        item["date"] = None
        item["prev_R"] = None
        item["prev_S"] = None
        item["prev_date"] = None
        item["next_date"] = str(now_date)
        review_list.append(item)
        learning_log = learning_log.append(item, ignore_index=True)

learning_log.to_csv("learning_log.csv", index=False)

In [ ]:
import asyncio, time
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

def display_item(item):
    future = asyncio.Future()
    
    clear_output()
    time.sleep(0.5)
    
    ref = wl[item["wl_name"]][wl[item["wl_name"]]["id"] == item["wl_id"]].iloc[0]
    
    title_lable = widgets.Label(value = r"\(\Large \bf \text{" + ref["word"] + "}\)")
    summary_lable = widgets.Label(value = r"\(\href{" + ref["url"] + "}{\\text{Definition}}\)")
    
    remember_button = widgets.Button(description = "Remember")
    def remember_on_click(change):
        future.set_result(True)
        remember_button.on_click(remember_on_click, remove=True)
    remember_button.on_click(remember_on_click)
    
    forget_button = widgets.Button(description = "Forget")
    def forget_on_click(change):
        future.set_result(False)
        forget_button.on_click(forget_on_click, remove=True)
    forget_button.on_click(forget_on_click)
    
    display_box =  widgets.VBox([title_lable, summary_lable, widgets.HBox([remember_button, forget_button])])
    display_box.layout.align_items = "center"
    
    display(display_box)
    return future

def process_score(score):
    for i in range(len(review_list)):
        item = review_list[i]
        new_R = score[i]
        new_S = initial_stability
        new_date = str(now_date)
        
        if isinstance(item["date"], str):
            item["S"] = -(now_date - datetime.date.fromisoformat(item["date"])).days / np.log(new_R)
            new_S = item["S"]
            if isinstance(item["prev_S"], float) and (item["prev_S"] < item["S"]):
                new_S = item["S"] * item["S"] / item["prev_S"]
        
        next_date = now_date + datetime.timedelta(days = 1 - np.log(retention) * new_S)
        
        ref = learning_log[(learning_log["wl_name"] == item["wl_name"]) &
                           (learning_log["wl_id"] == item["wl_id"])].index[0]
        learning_log.loc[ref, "prev_R"] = item["R"]
        learning_log.loc[ref, "prev_S"] = item["S"]
        learning_log.loc[ref, "prev_date"] = item["date"]
        learning_log.loc[ref, "R"] = new_R
        learning_log.loc[ref, "S"] = new_S
        learning_log.loc[ref, "date"] = new_date
        learning_log.loc[ref, "next_date"] = str(next_date)
    learning_log.to_csv("learning_log.csv", index=False)

def process_result(all_results):
    x = np.zeros(len(review_list))
    for result in all_results:
        x[~result] += 1
    score = 1 - x / len(all_results)
    score = np.minimum(score, 1 - 1e-6)
    score = np.maximum(score, 1e-6)
    process_score(score)

async def display_test():
    success = 0
    all_results = []
    
    while success < min_consecutive_success:
        result = np.full(len(review_list), False)
        random_index = np.random.permutation(len(review_list))
        
        for i in range(len(review_list)):
            result[random_index[i]] = await display_item(review_list[random_index[i]])
        
        all_results.append(result)
        if np.all(result):
            success = success + 1
        else:
            success = 0
    
    clear_output()
    time.sleep(0.5)
    process_result(all_results)
    print("Finish!")
    
asyncio.create_task(display_test())